# Bài 4 — Xử lý video: pipeline chuẩn có live preview

**Mục tiêu:** Nắm 3 utility video của supervision, viết được pipeline **vừa xem trực tiếp trên cửa sổ, vừa ghi file**.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [1]:
!pip install -q supervision ultralytics "supervision[assets]"

In [2]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 10:12:05] [INFO] supervision.assets.downloader - vehicles.mp4 asset download complete.
vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 4.1. Ba công cụ video

In [5]:
import supervision as sv

VIDEO_PATH = "vehicles.mp4"

# (1) Đọc thông tin video
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
print(video_info)  # width, height, fps, total_frames

# (2) Generator duyệt frame — thay cho vòng while cap.read() truyền thống
frame_generator = sv.get_video_frames_generator(VIDEO_PATH)
# Tùy chọn: get_video_frames_generator(VIDEO_PATH, stride=2, start=100, end=500)

# (3) Ghi video output — chỉ minh họa cú pháp (chạy thật ở 4.2)
# with sv.VideoSink(target_path="output.mp4", video_info=video_info) as sink:
#     for frame in frame_generator:
#         sink.write_frame(frame)

VideoInfo(width=3840, height=2160, fps=25.0, total_frames=538)


>  **Vì sao không dùng `sv.process_video`?** Hàm đó chạy "câm" — cắm mặt ghi file, không thấy gì cho tới khi xong. Ta tự viết vòng lặp để **vừa hiện cửa sổ vừa ghi file**, lại chủ động bấm Q dừng giữa chừng.

## 4.2.  Pipeline video hoàn chỉnh 

In [6]:
import numpy as np
from ultralytics import YOLO
from display import show_frame, close_windows

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "bai4_output.mp4"
VEHICLE_CLASSES = [2, 3, 5, 7]

model = YOLO("yolov8n.pt")
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)


def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[
        (detections.confidence > 0.3) & np.isin(detections.class_id, VEHICLE_CLASSES)
    ]

    labels = [f"{name} {conf:.2f}" for name, conf
              in zip(detections.data["class_name"], detections.confidence)]

    annotated = frame.copy()
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    return annotated

In [7]:
#  Vòng lặp chuẩn: XEM TRỰC TIẾP trên cửa sổ + GHI FILE song song
with sv.VideoSink(target_path=TARGET_VIDEO, video_info=video_info) as sink:
    for frame in sv.get_video_frames_generator(SOURCE_VIDEO):
        annotated = process_frame(frame)
        sink.write_frame(annotated)              # ghi file
        if not show_frame(annotated):            #  hiện cửa sổ; bấm Q -> dừng
            print("Nguoi dung bam Q - dung som.")
            break

close_windows()
print("Xong! Video da luu tai:", TARGET_VIDEO)

Xong! Video da luu tai: bai4_output.mp4


>  **Mẹo tăng tốc:** truyền `model(frame, imgsz=640, verbose=False)`; nếu có GPU thì `model.to("cuda")`. Với video 4K, dùng `imgsz=1280` cân bằng tốc độ/độ chính xác. Nếu preview giật vì máy yếu: thêm `stride=2` vào generator (bỏ qua 1 frame lấy 1 frame) — chỉ để học, khi xuất file thật thì bỏ stride.

Lưu ý: vòng lặp này chạy toàn bộ video (`wait=1` mặc định, không chặn) — bấm **Q** trên cửa sổ để dừng sớm bất cứ lúc nào.

##  Checkpoint Bài 4

Cửa sổ hiện video với box + label chạy mượt trực tiếp, bấm Q dừng được giữa chừng, và file `bai4_output.mp4` vẫn được ghi.

**Câu hỏi tự kiểm tra:** Vì sao dùng generator (`get_video_frames_generator`) thay vì đọc hết video vào RAM (ví dụ đọc hết vào một list trước rồi mới xử lý)?